# 🎮 Nova Launcher — Android APK Builder
**Qadamlar:**
1. Chapdan `Files` ikonkasini bosing
2. `nova_mobile` papkasidagi barcha fayllarni yuklang
3. Har bir cell ni ▶ tugmasi bilan ishga tushiring
4. Oxirida `bin/` papkasidan `.apk` faylni yuklab oling

## 📤 1-qadam: Fayllarni yuklang
Quyidagi cell ni ishga tushiring, keyin `nova_mobile` papkasidagi fayllarni yuklang:
- `main.py`
- `assets/logo.png`
- `assets/background.png`
- `buildozer.spec`

In [ ]:
import os
from google.colab import files

# Upload qilish uchun papka
os.makedirs('/content/nova_mobile/assets', exist_ok=True)
os.chdir('/content/nova_mobile')

print('✓ Papka tayyor!')
print('Endi fayllarni yuklang:')
uploaded = files.upload()

In [ ]:
# Yuklangan fayllarni to'g'ri joylarga ko'chirish
import shutil

for fname, data in uploaded.items():
    if fname in ['logo.png', 'background.png']:
        path = f'/content/nova_mobile/assets/{fname}'
    else:
        path = f'/content/nova_mobile/{fname}'
    with open(path, 'wb') as f:
        f.write(data)
    print(f'✓ {fname} saqlandi -> {path}')

print('\nPapka tarkibi:')
!find /content/nova_mobile -type f

## 🛠️ 2-qadam: Tizimni tayyorlash (10-15 daqiqa)

In [ ]:
%%bash
# Java o'rnatish
apt-get update -qq
apt-get install -y -qq openjdk-17-jdk
java -version
echo '✓ Java tayyor'

In [ ]:
%%bash
# Android SDK kerakli paketlar
apt-get install -y -qq \
    git \
    zip \
    unzip \
    autoconf \
    libtool \
    pkg-config \
    zlib1g-dev \
    libncurses5:i386 \
    libstdc++6:i386 \
    cmake \
    libffi-dev \
    libssl-dev
echo '✓ Kerakli paketlar o\'rnatildi'

In [ ]:
%%bash
# Python kutubxonalari
pip install -q buildozer cython==0.29.33
pip install -q kivy
echo '✓ Buildozer va Kivy o\'rnatildi'

## 📱 3-qadam: buildozer.spec konfiguratsiya

In [ ]:
spec_content = '''[app]
title = Nova Launcher
package.name = novalauncher
package.domain = com.nova.launcher
source.dir = .
source.include_exts = py,png,jpg,kv,atlas,json
source.include_patterns = assets/*
version = 2.5.0
requirements = python3,kivy
icon.filename = %(source.dir)s/assets/logo.png
orientation = portrait
fullscreen = 0
android.api = 33
android.minapi = 21
android.archs = arm64-v8a, armeabi-v7a
android.allow_backup = True
android.permissions = android.permission.INTERNET,android.permission.WRITE_EXTERNAL_STORAGE

[buildozer]
log_level = 2
warn_on_root = 1
'''

with open('/content/nova_mobile/buildozer.spec', 'w') as f:
    f.write(spec_content)

print('✓ buildozer.spec yozildi')

## 🔨 4-qadam: APK QURISH (30-60 daqiqa kutasiz!)

In [ ]:
import os
os.chdir('/content/nova_mobile')

# JAVA_HOME sozlash
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

print('📱 APK qurilmoqda... Bu 30-60 daqiqa davom etadi!')
print('Kutib turing...')
!buildozer -v android debug 2>&1 | tail -50

print('\n✅ Jarayon tugadi!')

# APK mavjudligini tekshirish
import glob
apks = glob.glob('/content/nova_mobile/bin/*.apk')
if apks:
    print(f'\n🎉 APK tayyor: {apks[0]}')
    print(f'Hajmi: {os.path.getsize(apks[0]) / 1024 / 1024:.1f} MB')
else:
    print('⚠ APK topilmadi. Xatolikni tekshiring.')

## 📥 5-qadam: APK ni yuklab olish

In [ ]:
from google.colab import files
import glob

apks = glob.glob('/content/nova_mobile/bin/*.apk')
if apks:
    print(f'✓ APK yuklab olinmoqda: {apks[0]}')
    files.download(apks[0])
    print('\n🎮 APK yuklab olindi!')
    print('Telefoningizda: Sozlamalar > Xavfsizlik > Noma\'lum manbalar')
    print('APK faylni ochib o\'rnating!')
else:
    print('❌ APK topilmadi. 4-qadam xatolarini tekshiring.')

## 🔧 Agar xato chiqsa — tez tuzatish

In [ ]:
# Qayta qurishdan oldin eski fayllarni tozalash
!buildozer android clean
print('✓ Tozalandi, qaytadan 4-qadam ni bosing')